In [2]:
def pad(s):
    # Un bloc AES fait 16 octets
    # Calcul du nombre d'octets pour obtenir la taille exacte nécessaire
    nb_octets = 16 - (len(s) % 16)
    
    # ça commence par 10000000 (0x80) suivi de 00000000 (0x00)
    padding = b'\x80' + b'\x00' * (nb_octets - 1)
    return s + padding


def unpad(s):
    # Vérification que la longueur totale est un multiple de 16 octets
    if len(s) == 0 or len(s) % 16 != 0:
        raise ValueError("message non correctement paddé")
    
    # Recherche du dernier octet non nul en partant de la fin
    i = len(s) - 1
    while i >= 0 and s[i] == 0:
        i -= 1
    
    # L'octet juste avant les zéros doit être 0x80 (128)
    if i < 0 or s[i] != 0x80:
        raise ValueError("message non correctement paddé")
    
    # retire le padding
    return s[:i]

In [3]:
# Tests avec différentes longueurs de chaînes
m1 = b'ceci est une bytestring'
m2 = b'test'
m3 = b'1234567890123456'  # Longueur exacte de 16 octets

# Test 1 : Ajout et retrait du padding
p1 = pad(m1)
p2 = pad(m2)
p3 = pad(m3)

print("Taille initiale m1:", len(m1), "-> Taille paddée:", len(p1))
print("Taille initiale m3:", len(m3), "-> Taille paddée:", len(p3))

assert unpad(p1) == m1
assert unpad(p2) == m2
assert unpad(p3) == m3
print("Tests de padding/unpadding : OK")

# Test 2 : Vérification de l'exception si le padding est incorrect
try:
    unpad(b'message_incorrect')
    print("Erreur : l'exception n'a pas été soulevée")
except ValueError as e:
    print("Exception bien attrapée :", e)

Taille initiale m1: 23 -> Taille paddée: 32
Taille initiale m3: 16 -> Taille paddée: 32
Tests de padding/unpadding : OK
Exception bien attrapée : message non correctement paddé


In [2]:
from hashlib import md5, sha1, sha3_256
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
from Crypto.Random.random import randrange
from Crypto.Util.number import bytes_to_long, long_to_bytes
from Crypto.Util.strxor import strxor as xor


def Enc(bloc: bytes, clef: bytes) -> bytes:
    if not isinstance(bloc, bytes) or not isinstance(clef, bytes):
        raise TypeError("le bloc et la clef doivent être de type bytes")
    if len(bloc) != 16 or len(clef) != 16:
        raise ValueError("le bloc et la clef doivent faire 16 octets")
    return AES.new(clef, AES.MODE_ECB).encrypt(bloc)


def Dec(bloc: bytes, clef: bytes) -> bytes:
    if not isinstance(bloc, bytes) or not isinstance(clef, bytes):
        raise TypeError("le bloc et la clef doivent être de type bytes")
    if len(bloc) != 16 or len(clef) != 16:
        raise ValueError("le bloc et la clef doivent faire 16 octets")
    return AES.new(clef, AES.MODE_ECB).decrypt(bloc)